# Batch Phase Fitting

Run `PhaseFitter` over a *sequence* of integrated patterns using the
`FitConfig` + `fit_sequence` + `FitResultStore` pipeline in
`ssrl_xrd_tools.analysis.fitting.batch`.

This is the headless equivalent of xdart's `BatchPhaseFitViewer`.
The result is a `pandas.DataFrame` with one row per pattern, ready
for plotting trends (phase fractions vs. position / temperature /
composition).


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from ssrl_xrd_tools.analysis.phase import PhaseModel
from ssrl_xrd_tools.analysis.fitting import FitConfig, fit_sequence

## ✏️ Configuration

**Edit the cell below**, then run the rest of the notebook top-to-bottom.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                  EDIT THIS CELL — paths + parameters                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── REQUIRED ───────────────────────────────────────────────────────────
data_dir     = Path('~/data/HZO_series').expanduser()  # ← REPLACE
patterns_pkl = data_dir / 'patterns.npz'                # ← REPLACE
cif_dir      = data_dir / 'cifs'                        # ← REPLACE

wavelength_A = 1.5406              # ← REPLACE — your beamline wavelength in Å

# ── OPTIONAL ───────────────────────────────────────────────────────────
out_csv  = data_dir / 'phase_fractions.csv'

# ── Fit window ─────────────────────────────────────────────────────────
q_range  = (1.5, 5.5)

### Validate the config

In [ ]:
assert data_dir.is_dir(), f'data_dir not found: {data_dir}'
assert patterns_pkl.is_file(), f'patterns_pkl not found: {patterns_pkl}'
assert cif_dir.is_dir(), f'cif_dir not found: {cif_dir}'
print(f'OK — patterns + CIF dir present.')

## Load patterns

`fit_sequence` accepts a list of ``(q, intensity)`` tuples (or
``(q, intensity, sigma)`` if you have per-point errors).  Here we
assume an `.npz` with arrays ``q``, ``intensities`` (shape
``(N_patterns, len(q))``) and optionally ``labels``.  Adapt this
cell if you're loading from NeXus, separate files, etc.

In [ ]:
npz = np.load(patterns_pkl, allow_pickle=True)
q              = np.asarray(npz['q'])
intensities    = np.asarray(npz['intensities'])      # (N, len(q))
labels         = list(npz['labels']) if 'labels' in npz else \
                 [f'frame_{i:04d}' for i in range(intensities.shape[0])]
patterns       = [(q, intensities[i]) for i in range(intensities.shape[0])]
print(f'Loaded {len(patterns)} patterns; q range '
      f'{q.min():.3f} → {q.max():.3f} Å⁻¹')

## Define phases

In [ ]:
phases = [
    PhaseModel.from_cif(cif_dir / 'phase_alpha.cif', name='alpha'),
    PhaseModel.from_cif(cif_dir / 'phase_beta.cif',  name='beta'),
]
for ph in phases:
    ph.calculate_peaks(wavelength=wavelength_A)
    print(f'  {ph.name}: {len(ph.peaks)} reflections (will be q-trimmed at fit time)')

## Build a FitConfig

`FitConfig` captures **two** keyword dicts:

- ``init_kw`` — passed to `PhaseFitter.__init__` (background settings,
  amorphous-peak settings, in-fit background, …).
- ``fit_kw`` — passed to `PhaseFitter.fit()` (q_range, profile,
  Caglioti toggles, texture, lattice/width bounds, …).

Plus ``phase_names`` (which phases from the list to include) and
``min_intensity`` (template-intensity floor for `add_phase`).  The
whole thing round-trips through JSON.

In [ ]:
config = FitConfig(
    init_kw={
        'prefit_background': 'snip',
    },
    fit_kw={
        'q_range': q_range,
        'method': 'leastsq',
        'lattice_pct': 0.05,           # ±5% lattice-parameter bounds
        'phase_profile': 'pseudovoigt',
    },
    phase_names=[ph.name for ph in phases],
    min_intensity=5.0,
    name='HZO series — first pass',
)
config.save(data_dir / 'fit_config.json')
print(f'Saved config → {data_dir / "fit_config.json"}')
# Reload elsewhere:  config = FitConfig.load(data_dir / 'fit_config.json')

## Run the sequence

In [ ]:
def _progress(i: int, n: int, result) -> None:
    if (i + 1) % 5 == 0 or i == n - 1:
        ok = 'OK' if result.success else 'FAIL'
        print(f'  [{i+1:3d}/{n}] redχ²={result.redchi:.3f}  {ok}')

store = fit_sequence(
    patterns,
    phases,
    config,
    labels=labels,
    progress_callback=_progress,
)
print(f'\nCompleted {len(store)} fits')

## Results as DataFrame

In [ ]:
df = store.to_dataframe()
print(df.head().to_string(index=False))
df.to_csv(out_csv, index=False)
print(f'\nSaved → {out_csv}')

## Plot phase fractions

In [ ]:
frac_cols = [c for c in df.columns if c.startswith('frac_')]

fig, ax = plt.subplots(figsize=(9, 4))
for col in frac_cols:
    ax.plot(df['index'], df[col], 'o-', label=col.replace('frac_', ''))
ax.set_xlabel('pattern index')
ax.set_ylabel('phase fraction')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout(); plt.show()

## (Optional) lattice-parameter trends

If `vary_cell_params=True` was set, the DataFrame also has columns
like `alpha_a`, `alpha_b`, `alpha_c`, etc.

In [ ]:
cell_cols = [c for c in df.columns
             if any(c.endswith('_' + x) for x in ('a', 'b', 'c'))]
if cell_cols:
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True)
    for ax, axis in zip(axes, ['a', 'b', 'c']):
        for ph_name in [p.name for p in phases]:
            col = f'{ph_name}_{axis}'
            if col in df.columns:
                ax.plot(df['index'], df[col], 'o-', label=ph_name)
        ax.set_ylabel(f'{axis} (Å)')
        ax.set_xlabel('pattern index')
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('No lattice-parameter columns — set vary_cell_params=True in FitConfig.')

---

### Re-running with new parameters

The whole pipeline is config-driven, so re-running with different
background / q_range / lattice flags is one config-edit + one
`fit_sequence` call.  The `FitResultStore` overwrites its rows on
re-fit; serialise it (`store.save(...)`) if you want to keep multiple
runs side-by-side.
